In [1]:
import os
import numpy as np
import torch

from BudaOCR.Encoder import WylieEncoder, StackEncoder
from BudaOCR.Networks import Easter2AttNetwork
from BudaOCR.Trainer import OCRTrainer

from BudaOCR.Config import CHARSET
from BudaOCR.Utils import (
    accumulate_distributions,
    build_data_paths,
    create_dir,
    read_stack_file,
    shuffle_data
    )

print(torch.__version__)
torch.cuda.empty_cache()

print(torch.cuda.is_available())

d:\Github\BDRC\tibetan-ocr-training_temp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.10.0+cu126
True


In [ ]:
image_width = 3200
image_height = 100
wylie_encoder = WylieEncoder(CHARSET)

stack_file = "tib-stacks_v2.txt"
stacks = read_stack_file(stack_file)
stack_encoder = StackEncoder(stacks)

# setting the encoder to be used
encoder = wylie_encoder

# train params
batch_size = 32
workers = 4
network = Easter2AttNetwork(image_width, image_height, num_classes=encoder.num_classes, easter_variant="fixed")

#### Single Dataset Training

In [3]:
# local dir
dataset_path = "D:/Datasets/KhyentseWangpo"
image_paths, label_paths = build_data_paths(dataset_path, img_file_ext="jpg")
image_paths, label_paths = shuffle_data(image_paths, label_paths)

print(f"Images: {len(image_paths)}, Labels: {len(label_paths)}")

output_dir = os.path.join("Output")
create_dir(output_dir)

Images: 13527, Labels: 13527


In [ ]:
ocr_trainer = OCRTrainer(
    network=network,
    label_encoder=encoder,
    workers=workers, 
    image_width=image_width,
    image_height=image_height,
    batch_size=batch_size, 
    output_dir=output_dir, 
    preload_labels=True
    )

ocr_trainer.init(image_paths, label_paths)

In [ ]:
num_epochs = 2
ocr_trainer.train(epochs=num_epochs, check_cer=True, export_onnx=True, silent=False)

#### Training multiple distributions

In [ ]:
data_root = "../home"
distributions = ["DergeTenjur", "LhasaKanjur", "Karmapa8", "LithangKanjur"]
distribution = accumulate_distributions(data_root, distributions)

In [ ]:
output_dir = "Output"
create_dir(output_dir)


ocr_trainer = OCRTrainer(
    network=network,
    label_encoder=wylie_encoder,
    workers=workers, 
    image_width=image_width,
    image_height=image_height,
    batch_size=batch_size, 
    output_dir=output_dir, 
    preload_labels=True
    )

assert (distribution is not None)
ocr_trainer.init_from_distribution(distribution)

In [ ]:
num_epochs = 12
ocr_trainer.train(epochs=num_epochs, check_cer=True, export_onnx=True, silent=False)

#### Output Dimension Debugging

In [ ]:
# debug start logits for non-zeros
test_sample = next(iter(ocr_trainer.test_loader))
test_logits, gt_labels = ocr_trainer.network.test(test_sample)
pred = np.argmax(test_logits[0], axis=0)
print(pred[:40]) # there should be no leading zeros